# ASL Alphabet Preprocessing Pipeline

This notebook implements the EDA-driven preprocessing pipeline for training a
MobileNetV2-based ASL alphabet recognition model.

- Image size: 128 × 128
- Grayscale images (replicated to 3 channels)
- MobileNetV2-compatible normalization
- Stratified train/validation split
- Reproducible preprocessing artifacts


#### Imports & Global Configuration

In [5]:
import os
import json
from pathlib import Path
from typing import Tuple, List

import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical


#### Configuration & Paths


In [6]:
# -----------------------------
# Global Configuration
# -----------------------------
IMAGE_SIZE = 128
NUM_CHANNELS = 3
RANDOM_STATE = 42

# -----------------------------
# Paths
# -----------------------------
RAW_DATA_DIR = Path("../data/raw/train")
PROCESSED_DATA_DIR = Path("../data/processed")

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)


#### Build Class-to-Index Mapping

In [7]:
# Discover class names from directory structure
class_names = sorted([
    d.name for d in RAW_DATA_DIR.iterdir()
    if d.is_dir()
])

num_classes = len(class_names)

class_to_index = {cls: idx for idx, cls in enumerate(class_names)}
index_to_class = {idx: cls for cls, idx in class_to_index.items()}

print(f"Detected {num_classes} classes:")
print(class_names)


Detected 29 classes:
['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']


In [8]:
# Persist label mappings for training and inference
with open(PROCESSED_DATA_DIR / "class_mapping.json", "w") as f:
    json.dump({
        "class_to_index": class_to_index,
        "index_to_class": index_to_class
    }, f, indent=4)


#### Image Preprocessing Function

In [9]:
def preprocess_image(image_path: Path) -> np.ndarray:
    """
    Load and preprocess a single image according to the pipeline contract.
    """
    # Load image (BGR)
    img = cv2.imread(str(image_path))
    if img is None:
        raise ValueError(f"Failed to read image: {image_path}")

    # Convert to grayscale
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Resize
    img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_LINEAR)

    # Normalize to [0, 1]
    img = img.astype(np.float32) / 255.0

    # MobileNetV2 normalization: [0,1] → [-1,1]
    img = (img * 2.0) - 1.0

    # Replicate channels: (H, W) → (H, W, 3)
    img = np.stack([img] * NUM_CHANNELS, axis=-1)

    return img


#### Load & Preprocess Dataset

In [10]:
X: List[np.ndarray] = []
y: List[int] = []

for class_name in class_names:
    class_dir = RAW_DATA_DIR / class_name
    label = class_to_index[class_name]

    for img_file in class_dir.iterdir():
        if img_file.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue

        try:
            img = preprocess_image(img_file)
            X.append(img)
            y.append(label)
        except Exception as e:
            print(f"Skipping {img_file}: {e}")

X = np.array(X)
y = np.array(y)

print(f"Dataset loaded: X={X.shape}, y={y.shape}")


KeyboardInterrupt: 

#### Stratified Split & One-Hot Encoding

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

y_train_oh = to_categorical(y_train, num_classes)
y_val_oh = to_categorical(y_val, num_classes)

print("Training set:", X_train.shape, y_train_oh.shape)
print("Validation set:", X_val.shape, y_val_oh.shape)


#### Save Processed Data

In [ ]:
np.save(PROCESSED_DATA_DIR / "X_train.npy", X_train)
np.save(PROCESSED_DATA_DIR / "X_val.npy", X_val)
np.save(PROCESSED_DATA_DIR / "y_train.npy", y_train_oh)
np.save(PROCESSED_DATA_DIR / "y_val.npy", y_val_oh)

print("Preprocessed data saved successfully.")


#### Save Preprocessing Summary JSON

In [ ]:
# -----------------------------
# Preprocessing Summary
# -----------------------------
preprocessing_summary = {
    "image_preprocessing": {
        "target_size": [IMAGE_SIZE, IMAGE_SIZE],
        "color_mode": "grayscale",
        "channels": NUM_CHANNELS,
        "channel_strategy": "grayscale_replicated",
        "interpolation": "bilinear"
    },
    "normalization": {
        "original_pixel_range": [0, 255],
        "scaled_range": [0, 1],
        "final_range": [-1, 1],
        "normalization_strategy": "mobilenetv2_compatible"
    },
    "labels": {
        "num_classes": num_classes,
        "class_names": class_names,
        "encoding": "integer + one-hot",
        "mapping_file": "data/processed/class_mapping.json"
    },
    "data_split": {
        "train_ratio": 0.8,
        "validation_ratio": 0.2,
        "stratified": True,
        "random_state": RANDOM_STATE
    },
    "design_rationale": {
        "grayscale": "Gesture recognition relies on hand shape, not color; improves robustness to lighting and skin-tone variation.",
        "resolution": "128x128 preserves hand geometry while enabling real-time inference performance.",
        "stratification": "Ensures balanced validation performance, especially for the 'nothing' class."
    }
}

# Save preprocessing summary
summary_path = Path("reports/preprocessing_summary.json")
summary_path.parent.mkdir(parents=True, exist_ok=True)

with open(summary_path, "w") as f:
    json.dump(preprocessing_summary, f, indent=4)

print(f"Preprocessing summary saved to: {summary_path}")
